# WellScope: Alberta oil facility screener (Whitecourt pilot)

**Who it's for:** a small buyer screening producing oil batteries and checking whether the seller's area cleanup liability could swamp the value.

**Everything is downloaded automatically. You upload nothing.** Cell 3 pulls the AER well list (about 170 MB) and cell 4 pulls 36 months of Petrinex volumes. Expect roughly 5 to 10 minutes for the first and 20 to 40 for the second. Colab keeps the files while the session is open, so re-running is fast.

**What it does:** fits decline curves with a *range* of shapes, values each battery (pre-tax PV10), back-tests the method on the last 6 months, places batteries on a map using nearby AER wells, and rolls everything up by operator with a cleanup-liability check. Exports Excel plus an interactive map page.

**What it is not:** reserves. Public data is battery-level (several wells combined), prices and costs below are placeholders, and cleanup cost is not AER's liability rating. Check AER and Petrinex terms of use before charging anyone (the AER ST37 page states no reuse licence; see aer.ca/copyright-and-disclaimer).

**How to run:** run each cell in order, waiting for it to finish.

### 1. Settings (every number here is an assumption you can change)

In [ ]:
END_MONTH = "2026-07"          # last Petrinex month to pull
N_MONTHS  = 36
TWP, RGE, MER = (55, 64), (6, 18), 5      # pilot area filter for Petrinex facilities (Whitecourt, W5)
WELL_LAT, WELL_LON = (53.4, 54.8), (-117.0, -114.6)   # AER wells kept for the map (wide enough to cover those townships/ranges)
AER_URL = "https://www.aer.ca/prd/documents/sts/st37/ST_37_Excel.zip"

# Placeholder economics, pre-tax. NOT data.
NETBACK      = 45.0     # $/bbl after differentials, transport and variable costs, BEFORE royalty
ROYALTY_RATE = 0.15     # flat placeholder (real Alberta royalty depends on price and well)
OPEX_BASE    = 6000.0   # $/month fixed opex per facility
OPEX_PER_BPD = 60.0     # $/month per bbl/d of current rate (bigger batteries cost more to run)
DISCOUNT     = 0.10     # PV10
B_GRID       = (0.3, 0.5, 0.8, 1.0)       # decline shapes tried; results reported as a range
CLEANUP_BASE, CLEANUP_PER_M = 80000, 55  # $ per active/suspended well: base + $/m depth (placeholder)
BACKTEST_MONTHS = 6
print("settings loaded")


### 2. Imports and the decline and economics engine

In [ ]:
import os, io, re, sys, json, zipfile, urllib.request, urllib.error, datetime as dt
import numpy as np, pandas as pd
from scipy.optimize import curve_fit

M3_TO_BBL = 6.2898

def arps(t, qi, di, b):
    """Arps hyperbolic: q(t) = qi / (1 + b*di*t)^(1/b); t in years, di nominal 1/yr."""
    return qi / np.power(1.0 + b * di * t, 1.0 / b)

def fit_fixed_b(rates, b, window=36, min_points=12):
    """Fit qi and di with b held fixed (b is poorly determined by short noisy series)."""
    y = np.asarray(rates, float)[-window:]
    if len(y) < min_points or y[-3:].min() <= 0:
        return None
    y = y[int(np.argmax(y)):]
    if len(y) < min_points:
        return None
    t = np.arange(len(y)) / 12.0
    m = y > 0
    try:
        p, _ = curve_fit(lambda t, qi, di: arps(t, qi, di, b), t[m], y[m], p0=[y[0], 0.5],
                         bounds=([0, 0.01], [y.max() * 2, 5.0]), maxfev=5000)
    except (RuntimeError, ValueError):
        return None
    pred = arps(t[m], p[0], p[1], b)
    ss_tot = np.sum((y[m] - y[m].mean()) ** 2) or 1.0
    tn = (len(y) - 1) / 12.0
    return dict(qi=p[0], di=p[1], b=b, r2=1 - np.sum((y[m] - pred) ** 2) / ss_tot,
                t_now=tn, q_now=float(arps(tn, p[0], p[1], b)))

def forecast(f, q_limit, d_min=0.06, max_months=360):
    """Monthly bbl/d from next month until rate < q_limit. Hyperbolic, then exponential terminal
    decline once the instantaneous decline falls to d_min (6%/yr nominal)."""
    qs, switched, q_term, m0 = [], False, None, 0
    for m in range(1, max_months + 1):
        t = f["t_now"] + m / 12.0
        if not switched and f["di"] / (1 + f["b"] * f["di"] * t) <= d_min:
            switched, q_term, m0 = True, arps(t, f["qi"], f["di"], f["b"]), m
        q = q_term * np.exp(-d_min * (m - m0) / 12.0) if switched else arps(t, f["qi"], f["di"], f["b"])
        if q < q_limit:
            break
        qs.append(q)
    return np.array(qs)

def economics(qs, opex_month):
    """Remaining oil (bbl) and PV10 ($, pre-tax) for a monthly rate path."""
    if len(qs) == 0:
        return 0.0, 0.0
    vol = qs * 30.4
    cash = vol * NETBACK * (1 - ROYALTY_RATE) - opex_month
    disc = (1 + DISCOUNT) ** (-(np.arange(1, len(qs) + 1)) / 12.0)
    return float(vol.sum()), float((cash * disc).sum())

def economic_limit(opex_month):
    """q_limit (bbl/d) = fixed opex per month / (net revenue per bbl * 30.4 days)."""
    return opex_month / (NETBACK * (1 - ROYALTY_RATE) * 30.4)
print("engine ready")


### 3. Download the AER well list and build the wells file (about 5 to 10 minutes)

In [ ]:
import glob, shutil
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def download(url, dest, chunk=1 << 20):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as f:
        total, done, last = int(r.headers.get("Content-Length") or 0), 0, 0
        while True:
            b = r.read(chunk)
            if not b: break
            f.write(b); done += len(b)
            if done - last > 25 * chunk:
                print(f"  {done/1e6:,.0f} MB" + (f" of {total/1e6:,.0f}" if total else ""), flush=True); last = done

def find_xlsx(root, key):
    hits = [p for p in glob.glob(root + "/**/*.xlsx", recursive=True)
            if key in os.path.basename(p).lower().replace(" ", "")]
    assert hits, f"Could not find a file containing '{key}' in the AER zip. Files: {os.listdir(root)}"
    return hits[0]

def clean(s):
    return s.astype(str).str.strip().replace({"None": None, "nan": None})

if not os.path.exists("wells_pilot.csv"):
    if not os.path.exists("ST_37_Excel.zip"):
        print("downloading AER ST37 Excel package ...", flush=True)
        download(AER_URL, "ST_37_Excel.zip")
    shutil.rmtree("st37", ignore_errors=True)
    with zipfile.ZipFile("ST_37_Excel.zip") as z: z.extractall("st37")
    print("reading surface hole coordinates ...", flush=True)
    sh = pd.read_excel(find_xlsx("st37", "st37_sh"), dtype={"Well_Licence_Number": str},
                       usecols=["Well_Licence_Number", "SH_Actual_Latitude", "SH_Actual_Longitude", "Licence_Surface_Location_Label"])
    sh["lic"] = clean(sh["Well_Licence_Number"])
    sh = sh[sh.SH_Actual_Latitude.between(*WELL_LAT) & sh.SH_Actual_Longitude.between(*WELL_LON)]
    sh = sh[["lic", "SH_Actual_Latitude", "SH_Actual_Longitude", "Licence_Surface_Location_Label"]]
    print(f"  {len(sh):,} wells inside the area box")
    print("reading production strings (status, fluid, depth). This is the slow one, a few minutes ...", flush=True)
    ps = pd.read_excel(find_xlsx("st37", "production_strings"), dtype={"Well_Licence_Number": str},
                       usecols=["Prod_String_UWI", "Well_Licence_Number", "Well_Name", "Licensee", "Status_Fluid", "Status_Mode",
                                "Licence_Status", "Final_Total_Depth", "Spud_Date"])
    ps["lic"] = clean(ps["Well_Licence_Number"])
    for c in ["Status_Fluid", "Status_Mode", "Well_Name", "Licensee", "Licence_Status"]:
        ps[c] = clean(ps[c])
    df = sh.merge(ps, on="lic", how="inner")
    out = df.rename(columns={"Prod_String_UWI": "uwi", "Well_Licence_Number": "licence_no", "Well_Name": "well_name",
                             "Licensee": "licensee", "Status_Fluid": "fluid", "Status_Mode": "mode", "Licence_Status": "licence_status",
                             "Final_Total_Depth": "total_depth_m", "Spud_Date": "spud_date",
                             "SH_Actual_Latitude": "lat", "SH_Actual_Longitude": "lon", "Licence_Surface_Location_Label": "location"})
    out = out[["uwi", "licence_no", "well_name", "licensee", "fluid", "mode", "licence_status", "total_depth_m", "spud_date", "lat", "lon", "location"]]
    out.to_csv("wells_pilot.csv", index=False)
    shutil.rmtree("st37", ignore_errors=True)
wells = pd.read_csv("wells_pilot.csv", low_memory=False)
print(f"wells ready: {len(wells):,} rows")
print(wells["mode"].value_counts().head(8).to_string())


### 4. Download Petrinex volumes (skipped if already downloaded in this session). About 20 to 40 minutes.

In [ ]:
URL = "https://www.petrinex.gov.ab.ca/publicdata/API/Files/AB/Vol/{m}/CSV"
KEEP = ["ProductionMonth","OperatorBAID","OperatorName","ReportingFacilityID","ReportingFacilityType",
        "ReportingFacilityName","ReportingFacilitySubTypeDesc","ReportingFacilityLocation","ActivityID",
        "ProductID","Volume","Hours","FacilityTownship","FacilityRange","FacilityMeridian"]

def read_bytes(data):
    while data[:2] == b"PK":                       # the download is zipped more than once
        z = zipfile.ZipFile(io.BytesIO(data)); data = z.read(z.namelist()[0])
    return data

def filter_month(data):
    df = pd.read_csv(io.BytesIO(read_bytes(data)), dtype=str, low_memory=False, encoding_errors="replace")
    missing = [c for c in KEEP if c not in df.columns]
    if missing:
        raise SystemExit(f"Columns not found: {missing}\nFile has: {list(df.columns)}")
    t, r, mer = (pd.to_numeric(df[c], errors="coerce") for c in ["FacilityTownship","FacilityRange","FacilityMeridian"])
    return df[t.between(*TWP) & r.between(*RGE) & (mer == MER)][KEEP]

if not os.path.exists("petrinex_area.csv"):
    os.makedirs("petrinex_area", exist_ok=True)
    months = pd.period_range(end=END_MONTH, periods=N_MONTHS, freq="M").astype(str)
    parts = []
    for m in months:
        small = f"petrinex_area/{m}.csv"
        if not os.path.exists(small):
            for attempt in range(2):
                try:
                    print("downloading", m, "...", flush=True)
                    req = urllib.request.Request(URL.format(m=m), headers={"User-Agent": "Mozilla/5.0"})
                    filter_month(urllib.request.urlopen(req, timeout=300).read()).to_csv(small, index=False)
                    break
                except (urllib.error.URLError, TimeoutError, zipfile.BadZipFile, EOFError) as e:
                    print("  problem:", e)
            else:
                print("  FAILED", m); continue
        parts.append(pd.read_csv(small, dtype=str))
    assert parts, "No months loaded"
    pd.concat(parts, ignore_index=True).to_csv("petrinex_area.csv", index=False)
raw = pd.read_csv("petrinex_area.csv", dtype=str, low_memory=False)
print(f"{len(raw):,} rows, {raw.ReportingFacilityID.nunique():,} facilities, months {raw.ProductionMonth.min()} to {raw.ProductionMonth.max()}")


### 5. Fit every oil facility, value it, and back-test the method

In [ ]:
oil = raw[(raw.ActivityID == "PROD") & (raw.ProductID == "OIL")].copy()
oil["month"] = pd.to_datetime(oil.ProductionMonth.astype(str).str[:7], errors="coerce")
oil["Volume"] = pd.to_numeric(oil.Volume, errors="coerce")
oil["Hours"] = pd.to_numeric(oil.Hours, errors="coerce")
oil = oil.dropna(subset=["month", "Volume"])
DATA_THROUGH = oil.month.max().strftime("%Y-%m")

rows, bt = [], []
for fid, g in oil.groupby("ReportingFacilityID"):
    s = g.groupby("month").agg(v=("Volume", "sum"), h=("Hours", "max")).asfreq("MS")
    s["v"] = s["v"].fillna(0)
    dim = s.index.days_in_month.values
    h = s["h"].fillna(0).values
    rate = s["v"].values * M3_TO_BBL / np.where(h > 0, np.minimum(h / 24, dim), dim)   # bbl per operating day
    meta = g.sort_values("month").iloc[-1]
    base = dict(facility_id=fid, name=meta.ReportingFacilityName, operator=meta.OperatorName,
                type=meta.ReportingFacilityType, location=meta.ReportingFacilityLocation,
                last_month=s.index[-1].strftime("%Y-%m"), months=len(s), q_last=round(float(rate[-1]), 1))
    fits = [(b, fit_fixed_b(rate, b)) for b in B_GRID]
    fits = [(b, f) for b, f in fits if f]
    if len(fits) < len(B_GRID):
        base["status"] = "not producing / too little data"; rows.append(base); continue
    q_now = float(np.median([f["q_now"] for _, f in fits]))
    opex = OPEX_BASE + OPEX_PER_BPD * q_now
    ql = economic_limit(opex)
    res = []
    for b, f in fits:
        qs = forecast(f, ql); vol, pv = economics(qs, opex); res.append((f, vol, pv, len(qs)))
    vols, pvs, lives = [r[1] for r in res], [r[2] for r in res], [r[3] for r in res]
    r2 = float(np.mean([r[0]["r2"] for r in res]))
    mid = next(f for b, f in fits if b == 0.5)
    reasons = [x for x, bad in [("poor fit", r2 < 0.6), ("<24 months", len(s) < 24),
                                ("at/below economic limit", max(vols) <= 0)] if bad]
    capped = max(lives) >= 360
    base.update(status="fitted", q_now=round(q_now, 1), eff_decline_pct=round((1 - np.exp(-mid["di"])) * 100),
                r2_mean=round(r2, 2), oil_low=round(min(vols)), oil_mid=round(float(np.median(vols))), oil_high=round(max(vols)),
                pv10_low=round(min(pvs)), pv10_mid=round(float(np.median(pvs))), pv10_high=round(max(pvs)),
                life_mid_yrs=round(float(np.median(lives)) / 12, 1), opex_month=round(opex), q_limit=round(ql, 1),
                confidence=("OK (30-yr cap)" if capped else "OK") if not reasons else "LOW (" + ", ".join(reasons) + ")")
    rows.append(base)
    # back-test: fit on data before the last BACKTEST_MONTHS, predict their average rate
    if len(rate) >= 24 + BACKTEST_MONTHS:
        f0 = fit_fixed_b(rate[:-BACKTEST_MONTHS], 0.5)
        if f0:
            pred = forecast(f0, 0.0, max_months=BACKTEST_MONTHS).mean()
            act = rate[-BACKTEST_MONTHS:].mean()
            if act > 0:
                bt.append((fid, base["confidence"].startswith("OK"), (pred - act) / act))

fac = pd.DataFrame(rows)
fac["_ok"] = fac["confidence"].astype(str).str.startswith("OK") if "confidence" in fac else False
fac = fac.sort_values(["_ok", "pv10_mid"], ascending=False, na_position="last").drop(columns="_ok").reset_index(drop=True)
print(fac.status.value_counts().to_string())
print("\nConfidence:\n", fac.confidence.fillna("n/a").str.split(" ").str[0].value_counts().to_string())

btd = pd.DataFrame(bt, columns=["facility_id", "ok", "err"])
def bt_line(d, label):
    if len(d) == 0: return f"{label}: no facilities with enough history"
    return (f"{label}: n={len(d)}, median error {d.err.median()*100:+.0f}% (bias), median absolute error "
            f"{d.err.abs().median()*100:.0f}%, within +/-25%: {(d.err.abs() <= .25).mean()*100:.0f}%")
BACKTEST_TEXT = [bt_line(btd, "All fitted facilities"), bt_line(btd[btd.ok], "High-confidence only")]
print(f"\nBACK-TEST: fit on data up to {BACKTEST_MONTHS} months before the end, then predict the average rate of the last {BACKTEST_MONTHS} months")
for t in BACKTEST_TEXT: print(" ", t)


### 6. Place facilities on the map using AER wells at the same location

In [ ]:
LOC = re.compile(r"^(?:(\d{1,2})-)?(\d{1,2})-(\d{1,3})-(\d{1,2})W(\d)$")
def parse_loc(s):
    m = LOC.match(str(s).strip().upper().replace(" ", ""))
    if not m: return None
    lsd, sec, twp, rge, mer = m.groups()
    return (int(lsd) if lsd else None, int(sec), int(twp), int(rge), int(mer))

w = wells.dropna(subset=["lat", "lon"]).copy()
w["p"] = w["location"].map(parse_loc)
w = w[w.p.notna()]
w["k_lsd"] = w.p.map(lambda p: p)
w["k_sec"] = w.p.map(lambda p: p[1:])
exact = w.groupby("k_lsd")[["lat", "lon"]].mean()
sect = w.groupby("k_sec")[["lat", "lon"]].mean()

def place(loc):
    p = parse_loc(loc)
    if p is None: return (np.nan, np.nan, "unplaced")
    if p[0] is not None and p in exact.index:
        r = exact.loc[[p]].iloc[0]; return (r.lat, r.lon, "wells at same LSD (about 0.5 km)")
    if p[1:] in sect.index:
        r = sect.loc[[p[1:]]].iloc[0]; return (r.lat, r.lon, "wells in same section (about 1 to 2 km)")
    return (np.nan, np.nan, "unplaced (no wells in that section)")

fitted = fac[fac.status == "fitted"].copy()
pl = fitted.location.map(place)
fitted["lat"], fitted["lon"], fitted["placement"] = [p[0] for p in pl], [p[1] for p in pl], [p[2] for p in pl]
print(fitted.placement.value_counts().to_string())


### 7. Operator rollup with a cleanup-liability check

In [ ]:
SUFFIX = re.compile(r"\b(LTD|LIMITED|INC|INCORPORATED|CORP|CORPORATION|ULC|CO|COMPANY|LP|PARTNERSHIP)\b\.?")
def norm(n):
    n = str(n).upper().replace("&", " AND ")
    n = re.sub(r"[^A-Z0-9 ]", " ", n)
    n = SUFFIX.sub(" ", n)
    return re.sub(r"\s+", " ", n).strip()

def grp(mode):
    m = str(mode).upper()
    if m.startswith("ABANDONED ZONE"): return "Zone abandoned"
    if m.startswith(("ABANDONED", "JUNKED")): return "Abandoned"
    if any(k in m for k in ("PUMPING", "FLOWING", "GAS LIFT", "ACTIVE")): return "Active"
    if "SUSPENDED" in m: return "Suspended"
    return "Other"

wells["grp"] = wells["mode"].map(grp)
wells["lic_key"] = wells["licensee"].map(norm)
wells["cleanup"] = np.where(wells.grp.isin(["Active", "Suspended"]),
                            CLEANUP_BASE + CLEANUP_PER_M * pd.to_numeric(wells.total_depth_m, errors="coerce").fillna(0), 0.0)
lic = wells.groupby("lic_key").agg(act=("grp", lambda s: (s == "Active").sum()), sus=("grp", lambda s: (s == "Suspended").sum()),
                                   ab=("grp", lambda s: (s == "Abandoned").sum()), liab=("cleanup", "sum"))

fitted["op_key"] = fitted.operator.map(norm)
okf = fitted[fitted.confidence.astype(str).str.startswith("OK")]
ops = fitted.groupby("op_key").agg(operator=("operator", "first"), facilities=("facility_id", "count"), oil_bpd=("q_now", "sum"))
ops["pv10_ok"] = okf.groupby("op_key").pv10_mid.sum()
ops["pv10_ok"] = ops.pv10_ok.fillna(0)
ops = ops.join(lic, how="left")
ops["pv10_over_cleanup"] = np.where(ops.liab > 0, ops.pv10_ok / ops.liab, np.nan)
ops = ops.sort_values("pv10_ok", ascending=False).reset_index(drop=True)
unmatched = ops.liab.isna().sum()
print(f"{len(ops)} operators; {unmatched} had no licensee-name match in your wells file (shown as blank)")
print(ops.head(10)[["operator","facilities","oil_bpd","pv10_ok","act","sus","ab","liab","pv10_over_cleanup"]].round(1).to_string(index=False))


### 8. Export to Excel

In [ ]:
today = dt.date.today().isoformat()
SOURCES = (f"Sources: AER ST37 well list (your download); Petrinex public Conventional Volumetric data through {DATA_THROUGH} "
           f"(facility level, gross volumes). Estimates by WellScope, generated {today}. Screening only, not reserves.")
assump = pd.DataFrame([
    ("Generated", today), ("Petrinex data through", DATA_THROUGH), ("Area filter", f"MER W{MER}, TWP {TWP}, RGE {RGE}"),
    ("Netback $/bbl (before royalty)", NETBACK), ("Royalty rate (flat)", ROYALTY_RATE),
    ("Opex base $/month", OPEX_BASE), ("Opex $/month per bbl/d", OPEX_PER_BPD), ("Discount rate", DISCOUNT),
    ("Decline shapes (b) tried", str(B_GRID)), ("Cleanup $/well", f"{CLEANUP_BASE} + {CLEANUP_PER_M} per m depth (active and suspended only)"),
    ("Back-test (all)", BACKTEST_TEXT[0]), ("Back-test (high confidence)", BACKTEST_TEXT[1]),
    ("Basis", "Facility (battery) volumes, gross, per operating day. Wells feeding a battery are not separated."),
    ("Values", "Pre-tax PV10 ranges across b = 0.3 to 1.0. Not reserves. Placeholder prices and costs."),
    ("Liability", "Placeholder per-well formula matched to operator by licensee name. Not AER's licensee liability rating."),
    ("Terms of use", "Check AER and Petrinex terms before commercial use."),
], columns=["Item", "Value"])
fac_out = fitted[["facility_id","name","operator","location","placement","confidence","q_now","eff_decline_pct","r2_mean",
                  "oil_low","oil_mid","oil_high","pv10_low","pv10_mid","pv10_high","life_mid_yrs","opex_month","q_limit","months","last_month"]]
ops_out = ops[["operator","facilities","oil_bpd","pv10_ok","act","sus","ab","liab","pv10_over_cleanup"]]
with pd.ExcelWriter("wellscope_screen.xlsx", engine="openpyxl") as xw:
    fac_out.to_excel(xw, sheet_name="Facilities", index=False); ops_out.to_excel(xw, sheet_name="Operators", index=False)
    assump.to_excel(xw, sheet_name="Assumptions", index=False)
    for ws in xw.book.worksheets:
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(48, max(10, max(len(str(c.value or "")) for c in col[:60]) + 2))
        ws.freeze_panes = "A2"
print("wrote wellscope_screen.xlsx")


### 8b. Load the page template

In [ ]:
# Page template for the map (do not edit). Cell 9 fills it with your data.
TEMPLATE = r'''<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.css"><script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.js"></script><title>WellScope – real AER data (Whitecourt pilot)</title><style>
:root{--bg:#f6f5f1;--card:#fff;--ink:#1d2327;--mut:#66707a;--line:#dedbd3;--acc:#0b6e69}
@media(prefers-color-scheme:dark){:root{--bg:#14181b;--card:#1d2328;--ink:#e8ecee;--mut:#98a3ab;--line:#2d353b;--acc:#4cc2ba}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.45 -apple-system,system-ui,sans-serif}
.w{max-width:960px;margin:0 auto;padding:14px}h1{font-size:20px;margin:0}.n{font-size:12px;color:var(--mut)}
.b{background:rgba(217,142,4,.15);border:1px solid #d98e04;border-radius:8px;padding:8px 10px;font-size:13px;margin:10px 0}
input{width:100%;padding:11px;border:1px solid var(--line);border-radius:10px;background:var(--card);color:var(--ink);font-size:16px}
.c{display:flex;gap:6px;flex-wrap:wrap;margin:10px 0}.ch{border:1px solid var(--line);background:var(--card);color:var(--ink);border-radius:99px;padding:6px 12px;font-size:13px;cursor:pointer}.ch.on{background:var(--acc);color:#fff;border-color:var(--acc)}
.cd{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:12px;margin-bottom:12px}
#cv{width:100%;height:440px;border:1px solid var(--line);border-radius:12px;display:block;z-index:0}
.r{display:flex;justify-content:space-between;gap:8px;padding:8px 0;border-bottom:1px solid var(--line);cursor:pointer}.r small{display:block;color:var(--mut)}
.g{display:grid;grid-template-columns:1fr 1fr;gap:10px}.k{color:var(--mut);font-size:12px}.v{font-weight:600}
h2{font-size:13px;margin:12px 0 6px;text-transform:uppercase;letter-spacing:.04em;color:var(--mut)}
table{width:100%;border-collapse:collapse;font-size:13px}td,th{padding:5px 4px;text-align:right;border-bottom:1px solid var(--line)}td:first-child,th:first-child{text-align:left}tr.l{cursor:pointer}
.d{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:6px}
</style></head><body><div class="w"><h1>WellScope <span class="n">Alberta · Whitecourt pilot</span></h1>
<div class="b"><b>Screening tool, not reserves.</b> Wells: AER ST37 (status, depth, licensee). Facilities: Petrinex public volumes (battery level, gross). Values are pre-tax PV10 ranges from placeholder prices and costs. Every figure shows its basis.</div>
<input id="q" type="search" placeholder="Search UWI, licensee, well name, location…"><div class="c" id="ch"></div>
<div id="cv"></div><div class="n" style="margin:4px 0 12px">Small dots = wells by status. Large circles = oil facilities (blue = reliable fit, grey = low confidence). Layers button (top right) = satellite.</div>
<div id="fp"></div><div id="pf"></div><div class="cd" id="ft"></div><div class="cd" id="ot"></div><div class="cd"><h2 style="margin-top:0">Licensees (current filter)</h2><div style="overflow-x:auto"><table id="lt"></table></div>
<div class="n">Est. liability = $80k + $55/m depth for wells with active or suspended status (placeholder).</div></div>
<div class="cd"><h2 style="margin-top:0" id="cnt"></h2><div id="ls"></div></div></div>
<script>
const D=__DATA__.map((a,i)=>({i,uwi:a[0],lic:a[1],fl:a[2],mode:a[3],depth:a[4],yr:a[5],lat:a[6],lon:a[7],name:a[8],loc:a[9]}));
const FAC=__FAC__,OPS=__OPS__,META=__META__;
let showF=true,fsel=null;
const esc=s=>String(s==null?"":s).replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const fm=n=>n==null?"–":(n>=1e6?(n/1e6).toFixed(2)+"M":Math.round(n/1e3)+"k");
const isOK=f=>String(f.conf).startsWith("OK");
function drawFac(){if(!showF)return;FAC.forEach((f,i)=>{if(f.lat==null)return;
 L.circleMarker([f.lat,f.lon],{radius:6+Math.min(12,Math.sqrt((f.q||0)/5)),color:"#111",weight:fsel===f?4:1.5,fillColor:isOK(f)?"#1f77b4":"#9aa3ab",fillOpacity:.85}).bindTooltip(esc(f.name)+" · "+Math.round(f.q||0)+" bbl/d").on("click",()=>pickF(i)).addTo(LG)})}
function ftable(){const rows=FAC.map((f,i)=>({f,i})).sort((a,b)=>(isOK(b.f)-isOK(a.f))||((b.f.pvmid||0)-(a.f.pvmid||0))).slice(0,25);
 $("ft").innerHTML=`<h2 style="margin-top:0">Oil facilities ranked by PV10 (pre-tax screening)</h2><div style="overflow-x:auto"><table><tr><th>Facility</th><th>Operator</th><th>bbl/d</th><th>Decl/yr</th><th>PV10 $M (low–high)</th><th>Fit</th></tr>`+rows.map(({f,i})=>`<tr class="l" onclick="pickF(${i})"><td>${esc(f.name)}</td><td>${esc(f.op)}</td><td>${Math.round(f.q||0)}</td><td>${f.dec}%</td><td>${(f.pvlow/1e6).toFixed(1)}–${(f.pvhigh/1e6).toFixed(1)}</td><td>${isOK(f)?"OK":"low"}</td></tr>`).join("")+`</table></div><div class="n">Tap a row to open it on the map. "low" = poor fit or too little history, so treat as unreliable.</div>`}
function otable(){$("ot").innerHTML=`<h2 style="margin-top:0">Operators: facility value vs area cleanup liability</h2><div style="overflow-x:auto"><table><tr><th>Operator</th><th>Oil bbl/d</th><th>PV10 $M (OK fits)</th><th>Wells act/susp/aband</th><th>Cleanup $M (est.)</th><th>PV10 ÷ cleanup</th></tr>`+OPS.map(o=>`<tr><td>${esc(o.op)}</td><td>${Math.round(o.q)}</td><td>${(o.pv/1e6).toFixed(1)}</td><td>${o.act==null?"–":o.act+"/"+o.sus+"/"+o.ab}</td><td>${o.liab==null?"–":(o.liab/1e6).toFixed(1)}</td><td>${o.ratio==null?"–":o.ratio.toFixed(1)}</td></tr>`).join("")+`</table></div><div class="n">Cleanup = placeholder per-well formula on active and suspended wells in this area, matched by licensee name. It is not AER's liability rating. A ratio under 1 means area cleanup could exceed the value of these facilities.</div>`}
function pickF(i){const f=FAC[i];fsel=f;sel=null;pf();map();if(f.lat!=null&&M)M.setView([f.lat,f.lon],Math.max(M.getZoom(),11));
 const kv=(k,v)=>`<div><div class="k">${k}</div><div class="v">${v==null||v===""?"–":v}</div></div>`;
 $("fp").innerHTML=`<div class="cd" id="fc"><div class="n">Oil facility (battery level, not a single well)</div><div style="font-size:18px;font-weight:700">${esc(f.name)}</div>
 <h2>Identity</h2><div class="g">${kv("Operator",esc(f.op))}${kv("Facility ID",f.id)}${kv("Legal location",f.loc)}${kv("Map placement",f.prec)}</div>
 <h2>Production and decline</h2><div class="g">${kv("Current rate",Math.round(f.q)+" bbl/d (per operating day)")}${kv("Decline",f.dec+"%/yr (effective)")}${kv("Fit quality (avg r²)",f.r2)}${kv("Confidence",esc(f.conf))}</div>
 <h2>Value (screening, pre-tax)</h2><div class="g">${kv("Remaining oil",fm(f.olow)+" to "+fm(f.ohigh)+" bbl")}${kv("PV10",fm(f.pvlow)+" to "+fm(f.pvhigh))}${kv("Est. remaining life",f.life+" yrs")}${kv("Economic limit",Math.round(f.ql)+" bbl/d")}</div>
 <div class="n">Range comes from decline shape b = 0.3 to 1.0. Public volumes are for the whole battery. The wells feeding it are not separated.</div></div>`;
 $("fc").scrollIntoView({behavior:"smooth"})}
const grp=m=>/^(ABANDONED ZONE)/.test(m)?"Zone abandoned":/^(ABANDONED|JUNKED)/.test(m)?"Abandoned":/PUMPING|FLOWING|GAS LIFT|ACTIVE/.test(m)?"Active":/SUSPENDED/.test(m)?"Suspended":"Other";
const fl=f=>f==="CRUDE OIL"?"Oil":/GAS|COALBED/.test(f)?"Gas":"Other";
const COL={Active:"#2e8b57",Suspended:"#d98e04",Abandoned:"#b3423a","Zone abandoned":"#8a6bbf",Other:"#7a848c"};
D.forEach(w=>{w.g=grp(w.mode);w.f=fl(w.fl);w.liab=(w.g==="Active"||w.g==="Suspended")?80000+55*w.depth:0});
const F={g:"All",f:"All"};let sel=null,cur=[];
const $=id=>document.getElementById(id),fmt=n=>Math.round(n).toLocaleString("en-CA"),mon=n=>n>=1e6?"$"+(n/1e6).toFixed(1)+"M":"$"+fmt(n/1e3)+"k";
function chips(){let h="";[["g",["All","Active","Suspended","Abandoned","Zone abandoned","Other"]],["f",["All","Oil","Gas","Other"]]].forEach(([k,o])=>o.forEach(v=>h+=`<button class="ch ${F[k]===v?"on":""}" onclick="F.${k}='${v}';draw()">${v==="All"?(k==="g"?"All status":"All fluids"):v}</button>`));h+=`<button class="ch ${showF?"on":""}" onclick="showF=!showF;draw()">Facilities</button>`;$("ch").innerHTML=h}
function filt(){const q=$("q").value.toLowerCase();return D.filter(w=>(F.g==="All"||w.g===F.g)&&(F.f==="All"||w.f===F.f)&&(!q||(w.uwi+w.lic+w.name+w.loc).toLowerCase().includes(q)))}
let M,LG;
function initMap(){M=L.map("cv",{preferCanvas:true});
const E="https://server.arcgisonline.com/ArcGIS/rest/services/",st=L.tileLayer(E+"World_Street_Map/MapServer/tile/{z}/{y}/{x}",{maxZoom:19,attribution:"Esri"}),
sat=L.tileLayer(E+"World_Imagery/MapServer/tile/{z}/{y}/{x}",{maxZoom:18,attribution:"Esri, Maxar"}),
topo=L.tileLayer(E+"World_Topo_Map/MapServer/tile/{z}/{y}/{x}",{maxZoom:18,attribution:"Esri"});
st.addTo(M);L.control.layers({"Street":st,"Satellite":sat,"Topo":topo}).addTo(M);LG=L.layerGroup().addTo(M);
M.fitBounds([[Math.min(...D.map(w=>w.lat)),Math.min(...D.map(w=>w.lon))],[Math.max(...D.map(w=>w.lat)),Math.max(...D.map(w=>w.lon))]])}
function map(){LG.clearLayers();cur.forEach(w=>L.circleMarker([w.lat,w.lon],{radius:4,color:COL[w.g],weight:1,fillColor:COL[w.g],fillOpacity:.8}).on("click",()=>pick(w.i,false)).addTo(LG));
if(sel)L.circleMarker([sel.lat,sel.lon],{radius:11,color:"#000",weight:3,fill:false}).addTo(LG);drawFac()}
function roll(){const m={};cur.forEach(w=>{const o=m[w.lic]||(m[w.lic]={n:0,a:0,s:0,b:0,l:0});o.n++;if(w.g==="Active")o.a++;if(w.g==="Suspended")o.s++;if(w.g==="Abandoned")o.b++;o.l+=w.liab});
const rows=Object.entries(m).sort((a,b)=>b[1].n-a[1].n).slice(0,12);
$("lt").innerHTML="<tr><th>Licensee</th><th>Wells</th><th>Active</th><th>Susp.</th><th>Aband.</th><th>Est. liab.</th></tr>"+rows.map(([k,o])=>`<tr class="l" onclick="$('q').value=this.dataset.k;draw()" data-k="${k.replace(/"/g,"&quot;")}"><td>${k}</td><td>${o.n}</td><td>${o.a}</td><td>${o.s}</td><td>${o.b}</td><td>${mon(o.l)}</td></tr>`).join("")}
function draw(){chips();cur=filt();map();roll();$("cnt").textContent=fmt(cur.length)+" wells";
$("ls").innerHTML=cur.slice(0,60).map(w=>`<div class="r" onclick="pick(${w.i})"><div><span class="d" style="background:${COL[w.g]}"></span><b>${w.uwi}</b><small>${w.lic} · ${w.f}</small></div><div style="text-align:right">${w.g}<small>${w.mode}</small></div></div>`).join("")+(cur.length>60?`<div class="n" style="padding:8px">Showing 60 of ${fmt(cur.length)}. Search or filter to narrow.</div>`:"");pf();ftable();otable()}
function pf(){if(!sel){$("pf").innerHTML="";return}const w=sel,kv=(k,v)=>`<div><div class="k">${k}</div><div class="v">${v||"–"}</div></div>`;
const nb=D.filter(o=>o.i!==w.i).map(o=>({o,d:Math.hypot((o.lat-w.lat)*111,(o.lon-w.lon)*65)})).sort((a,b)=>a.d-b.d).slice(0,5);
const same=D.filter(o=>o.lic===w.lic),act=same.filter(o=>o.g==="Active").length,sus=same.filter(o=>o.g==="Suspended").length;
$("pf").innerHTML=`<div class="cd" id="pc"><div class="n">Well profile</div><div style="font-size:18px;font-weight:700">${w.uwi}</div>
<h2>Identity</h2><div class="g">${kv("Licensee",w.lic)}${kv("Well name",w.name)}${kv("Surface location",w.loc)}${kv("Fluid (AER)",w.fl)}${kv("Total depth",w.depth?fmt(w.depth)+" m":"")}${kv("Spud year",w.yr||"")}</div>
<a href="https://www.google.com/maps/search/?api=1&query=${w.lat},${w.lon}" target="_blank" style="color:var(--acc)">Open in Google Maps ↗</a><h2>Status</h2><div class="g">${kv("Group",`<span class="d" style="background:${COL[w.g]}"></span>${w.g}`)}${kv("AER mode",w.mode)}</div><div class="n">Status only. Whether it is actually flowing needs Petrinex volumes (next step).</div>
<h2>Liability (placeholder)</h2><div class="g">${kv("Est. cleanup cost",w.liab?"$"+fmt(w.liab):"n/a (abandoned or not applicable)")}${kv("Licensee in area",fmt(same.length)+" wells · "+act+" active · "+sus+" suspended")}</div>
<h2>Nearest wells</h2>${nb.map(n=>`<div><a href="#" style="color:var(--acc)" onclick="pick(${n.o.i});return false">${n.o.uwi}</a> <span class="n">${n.d.toFixed(1)} km · ${n.o.g} · ${n.o.lic}</span></div>`).join("")}</div>`}
function pick(i,sc=true){sel=D[i];fsel=null;$("fp").innerHTML="";map();pf();if(sc){M.setView([sel.lat,sel.lon],Math.max(M.getZoom(),13));$("cv").scrollIntoView({behavior:"smooth"})}}
$("q").addEventListener("input",draw);initMap();draw();document.querySelector(".w").insertAdjacentHTML("beforeend","<p class='n'>"+esc(META.sources)+"</p>");
</script></body></html>'''


### 9. Build and show the interactive map

In [ ]:
def clean(v):
    if v is None: return None
    if isinstance(v, (float, np.floating)):
        return None if not np.isfinite(v) else float(v)
    if isinstance(v, (np.integer,)): return int(v)
    return v

w2 = wells.dropna(subset=["lat", "lon"]).copy()
w2["yr"] = pd.to_datetime(w2["spud_date"], errors="coerce").dt.year.fillna(0).astype(int)
w2 = w2.fillna("")
WELL_ROWS = [[r.uwi, r.licensee, r.fluid, r["mode"], float(pd.to_numeric(r.total_depth_m, errors="coerce") or 0), int(r.yr),
              round(float(r.lat), 5), round(float(r.lon), 5), r.well_name, r.location] for _, r in w2.iterrows()]

FAC_ROWS = [dict(id=r.facility_id, name=r["name"], op=r.operator, loc=r.location, prec=r.placement, lat=clean(r.lat), lon=clean(r.lon),
                 q=clean(r.q_now), dec=clean(r.eff_decline_pct), r2=clean(r.r2_mean), conf=r.confidence,
                 olow=clean(r.oil_low), ohigh=clean(r.oil_high), pvlow=clean(r.pv10_low), pvmid=clean(r.pv10_mid),
                 pvhigh=clean(r.pv10_high), life=clean(r.life_mid_yrs), ql=clean(r.q_limit)) for _, r in fitted.iterrows()]
OPS_ROWS = [dict(op=r.operator, q=clean(r.oil_bpd) or 0, pv=clean(r.pv10_ok) or 0,
                 act=None if pd.isna(r.act) else int(r.act), sus=None if pd.isna(r.sus) else int(r.sus),
                 ab=None if pd.isna(r.ab) else int(r.ab), liab=clean(r.liab), ratio=clean(r.pv10_over_cleanup))
            for _, r in ops.head(25).iterrows()]
META = dict(sources=SOURCES)

html = (TEMPLATE.replace("__DATA__", json.dumps(WELL_ROWS, separators=(",", ":")))
                .replace("__FAC__", json.dumps(FAC_ROWS, separators=(",", ":"), allow_nan=False))
                .replace("__OPS__", json.dumps(OPS_ROWS, separators=(",", ":"), allow_nan=False))
                .replace("__META__", json.dumps(META)))
open("wellscope_map.html", "w").write(html)
print(f"wrote wellscope_map.html: {len(WELL_ROWS):,} wells, {len(FAC_ROWS)} facilities, "
      f"{sum(1 for f in FAC_ROWS if f['lat'] is not None)} placed on the map")
if IN_COLAB:
    from IPython.display import HTML, display
    display(HTML(html))


### 10. Download your files

In [ ]:
if IN_COLAB:
    files.download("wellscope_screen.xlsx"); files.download("wellscope_map.html")
    files.download("petrinex_area.csv"); files.download("wells_pilot.csv")
else:
    print("not in Colab: files are in the current folder")
